In [12]:
from utils.generator import Generator
from utils.mean_subtract import MeanSubtract

from keras.models import Sequential
from keras.layers import BatchNormalization
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dropout
from keras.layers.core import Dense

from keras.optimizers import Adam
from keras.callbacks import TensorBoard

import pickle

In [13]:
class SmallerVGGNet:
    @staticmethod
    def build(width, height, depth):
        model = Sequential()
        inputShape = (height, width, depth)
        
        model.add(Conv2D(32, 3, padding="same", input_shape=inputShape))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(MaxPooling2D(pool_size=3))
        model.add(Dropout(0.25))
        
        model.add(Conv2D(64, 3, padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(Conv2D(64, 3, padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(MaxPooling2D(pool_size=2))
        model.add(Dropout(0.25))
        
        model.add(Conv2D(128, 3, padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(Conv2D(128, 3, padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(MaxPooling2D(pool_size=2))
        model.add(Dropout(0.25))
        
        model.add(Flatten())
        model.add(Dense(1024))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=-1))
        model.add(Dropout(0.5))
        
        model.add(Dense(4))
        
        return model

In [14]:
width = 96
height = 96
depth = 3
BS = 128
epochs = 25

In [15]:
model = SmallerVGGNet.build(width, height, depth)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_5 (Conv2D)           (None, 96, 96, 32)        896       
                                                                 
 activation_6 (Activation)   (None, 96, 96, 32)        0         
                                                                 
 batch_normalization_6 (Batc  (None, 96, 96, 32)       128       
 hNormalization)                                                 
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 32, 32, 32)       0         
 2D)                                                             
                                                                 
 dropout_4 (Dropout)         (None, 32, 32, 32)        0         
                                                                 
 conv2d_6 (Conv2D)           (None, 32, 32, 64)       

### Training on extra dataset (pretrain)

In [16]:
means = pickle.load(open("rgb_means.pkl", "rb"))
ms = MeanSubtract(means["R"], means["G"], means["B"])
preprocessors = [ms]

# generator = Generator("bbox/extra.hdf5", BS, epochs, preprocessors)

In [17]:
opt = Adam(lr=0.001)
model.compile(loss="mean_squared_error", optimizer=opt)

In [18]:
H = model.fit_generator(generator.generate(), steps_per_epoch=generator.n_img//BS, epochs=20)

NameError: name 'generator' is not defined

### Training on training set

In [19]:
import numpy as np

def exp_decay(epoch):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * np.exp(-k*epoch)
    
    return lrate

In [20]:
from keras.callbacks import LearningRateScheduler
train_gen = Generator("bbox/train.hdf5", BS, epochs, preprocessors)
val_gen = Generator("bbox/val.hdf5", BS, epochs, preprocessors)

callbacks = [LearningRateScheduler(exp_decay)]

In [21]:
H = model.fit_generator(train_gen.generate(), validation_data=val_gen.generate(), 
                        steps_per_epoch=train_gen.n_img//BS, validation_steps=val_gen.n_img//BS,
                       epochs=epochs, callbacks=callbacks)

Epoch 1/25


C:\Users\mehrdad hooshangi\AppData\Local\Temp\ipykernel_23848\2875300423.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  H = model.fit_generator(train_gen.generate(), validation_data=val_gen.generate(),


241/241 [==============================] - 11s 43ms/step - loss: 1480.4861 - val_loss: 185.3582 - lr: 0.0010
Epoch 2/25
241/241 [==============================] - 10s 42ms/step - loss: 185.7657 - val_loss: 163.4040 - lr: 9.0484e-04
Epoch 3/25
241/241 [==============================] - 10s 42ms/step - loss: 162.4204 - val_loss: 137.4034 - lr: 8.1873e-04
Epoch 4/25
241/241 [==============================] - 10s 42ms/step - loss: 146.3890 - val_loss: 139.1602 - lr: 7.4082e-04
Epoch 5/25
241/241 [==============================] - 11s 44ms/step - loss: 137.3939 - val_loss: 132.6069 - lr: 6.7032e-04
Epoch 6/25
241/241 [==============================] - 10s 42ms/step - loss: 128.3985 - val_loss: 127.0490 - lr: 6.0653e-04
Epoch 7/25
241/241 [==============================] - 11s 44ms/step - loss: 121.9780 - val_loss: 127.9641 - lr: 5.4881e-04
Epoch 8/25
241/241 [==============================] - 10s 42ms/step - loss: 116.7542 - val_loss: 145.3761 - lr: 4.9659e-04
Epoch 9/25
241/241 [==========

In [22]:
model.save("detection.h5")